In [ ]:
#setup environment
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("Environment setup successful!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-Learn version: {sklearn.__version__}")

In [ ]:
#loading dataset
df = pd.read_csv("insurance.csv")
print(df.head(5))
print(df.shape) #shape is (1338, 7) which means there are 1338 rows and 7 columns
print(df.isnull().sum()) #checking for missing values

In [ ]:
plt.hist(df["charges"], bins=30)
plt.xlabel("Charges")
plt.ylabel("Frequency")
plt.title("Distribution of Charges")
plt.show()

In [ ]:
df = pd.get_dummies(
    df,
    columns=['sex', 'smoker', 'region'],
    drop_first=True
)

In [ ]:
df["sex_male"] = df["sex_male"].astype(int)
df["smoker_yes"] = df["smoker_yes"].astype(int)
df["region_northwest"] = df["region_northwest"].astype(int)
df["region_southeast"] = df["region_southeast"].astype(int)
df["region_southwest"] = df["region_southwest"].astype(int)

print(df.head(5))
print(df.shape) #shape is (1338, 9)

In [ ]:
plt.scatter(df["age"], df["charges"], c=df["smoker_yes"], cmap="coolwarm")
plt.xlabel("Age")
plt.ylabel("Charges")
plt.title("Age vs Charges colored by Smoker Status")
plt.legend(["Smoker"])
plt.show()
plt.scatter(df["bmi"], df["charges"], c=df["smoker_yes"], cmap="coolwarm")
plt.xlabel("BMI")
plt.ylabel("Charges")
plt.title("BMI vs Charges colored by Smoker Status")
plt.legend(["Smoker"])
plt.show()


In [ ]:
#splitting data into training and testing sets
y = df["charges"]
X = df.drop("charges", axis=1)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#feature scaling using z-score normalization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#training a linear regression model
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = lr_model.predict(X_test_scaled)
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R^2 Score: {r2}")

In [ ]:
#exploring polynomial features to capture non-linear relationships
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(X_train_scaled)
print(f"Original number of features: {X_train_scaled.shape[1]}")
print(f"Number of features after polynomial transformation: {poly_features.shape[1]}")

In [ ]:
lr_poly_model = LinearRegression()
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)
lr_poly_model.fit(X_train_poly, y_train)
y_poly_pred = lr_poly_model.predict(X_test_poly)
r2_train_poly = r2_score(y_train, lr_poly_model.predict(X_train_poly))
print(f"Polynomial R^2 Score on Training Set: {r2_train_poly}")
r2_test_poly = r2_score(y_test, y_poly_pred)
print(f"Polynomial R^2 Score on Test Set: {r2_test_poly}")

In [ ]:
from sklearn.linear_model import Ridge
ridge_model = Ridge(alpha=10.0)
ridge_model.fit(X_train_poly, y_train)
y_ridge_pred = ridge_model.predict(X_test_poly)
r2_ridge = r2_score(y_test, y_ridge_pred)
print(f"Ridge Regression R^2 Score on Test Set: {r2_ridge}")

In [ ]:
coefficients = ridge_model.coef_
feature_names = poly.get_feature_names_out(X.columns)
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})
print(coef_df.sort_values(by="Coefficient", ascending=False))

In [ ]:
residuals = y_test - y_ridge_pred
plt.figure(figsize=(8, 6))
plt.scatter(y_ridge_pred, residuals, alpha=0.8, edgecolor='k', linewidth=0.5)
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.xlabel("Predicted Charges")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Charges (Final Ridge Model)")
plt.grid(True)
plt.show()

In [ ]:
import joblib
joblib.dump(ridge_model, "insurance_model.pkl")
joblib.dump(scaler, "scaler.pkl")